# Notebook 07b — DaTSCAN Screening-Time Feature Verification

## Objective
Verify whether DaTSCAN/SBR tabular variables can be used as baseline/pre-baseline predictors for the PPMI primary analytic cohort.

## Scientific Background
The previous Notebook 07 found that DaTSCAN data are mostly stored under screening visit (`SC`) rather than baseline (`BL`). This notebook checks `SC` DaTSCAN overlap, missingness, and creates a clean DaTSCAN feature matrix without fitting any ML model.

## Expected Output
A verified DaTSCAN/SBR feature inventory for use in the next multimodal modeling stage.


In [ ]:
# ============================================================
# 01. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ============================================================
# 02. Imports and project paths
# ============================================================

from pathlib import Path
import os
import zipfile
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/PPMI_PD_Progression")

ADDITIONAL_DIRS = [
    PROJECT_DIR / "data" / "raw" / "additional",
    PROJECT_DIR / "data" / "additional",
]

EXTRACT_DIR = PROJECT_DIR / "data" / "extracted" / "imaging_selected"
NB02_DIR = PROJECT_DIR / "outputs" / "notebook_02_cohort_outcome"
OUT_DIR = PROJECT_DIR / "outputs" / "notebook_07b_datscan_screening_inventory"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("EXTRACT_DIR:", EXTRACT_DIR)
print("NB02_DIR exists:", NB02_DIR.exists())
print("OUT_DIR:", OUT_DIR)


In [ ]:
# ============================================================
# 03. Locate and extract imaging ZIP
# ============================================================

zip_candidates = []
for d in ADDITIONAL_DIRS:
    print("Checking:", d, "| exists:", d.exists())
    if d.exists():
        zip_candidates.extend(list(d.glob("*Imaging*Tabular*Selected*.zip")))
        zip_candidates.extend(list(d.glob("*imaging*tabular*selected*.zip")))
        zip_candidates.extend(list(d.glob("*DATSCAN*.zip")))
        zip_candidates.extend(list(d.glob("*DaTSCAN*.zip")))
        zip_candidates.extend(list(d.glob("*.zip")))

zip_candidates = sorted(set(zip_candidates))

print("\nZIP candidates found:")
for z in zip_candidates:
    print("-", z)

if not zip_candidates:
    raise FileNotFoundError("No imaging ZIP found. Place the imaging tabular ZIP in data/raw/additional or data/additional.")

IMAGING_ZIP = zip_candidates[0]
print("\nUsing imaging ZIP:", IMAGING_ZIP)

with zipfile.ZipFile(IMAGING_ZIP, "r") as z:
    z.extractall(EXTRACT_DIR)

csv_files = sorted(EXTRACT_DIR.rglob("*.csv"))
print("\nCSV files extracted:", len(csv_files))
for f in csv_files:
    print("-", f.name)


In [ ]:
# ============================================================
# 04. Load primary analytic cohort from Notebook 02
# ============================================================

cohort_candidates = [
    NB02_DIR / "11_primary_analytic_cohort_recommended_window.csv",
    NB02_DIR / "primary_analytic_cohort_recommended_window.csv",
]

cohort_path = None
for p in cohort_candidates:
    if p.exists():
        cohort_path = p
        break

if cohort_path is None:
    all_candidates = list(NB02_DIR.rglob("*analytic*cohort*.csv"))
    if all_candidates:
        cohort_path = all_candidates[0]

if cohort_path is None:
    raise FileNotFoundError("Primary analytic cohort file not found in Notebook 02 outputs.")

cohort = pd.read_csv(cohort_path)
if "PATNO" not in cohort.columns:
    raise ValueError("PATNO not found in primary analytic cohort.")

cohort["PATNO"] = cohort["PATNO"].astype(str)
primary_patnos = set(cohort["PATNO"])

print("Cohort file:", cohort_path.name)
print("Primary analytic cohort n:", len(cohort))
print("Unique PATNO:", cohort["PATNO"].nunique())


In [ ]:
# ============================================================
# 05. Load imaging files and identify DaTSCAN/SBR tables
# ============================================================

datasets = {}
inventory_rows = []

for f in csv_files:
    name = f.stem
    df = pd.read_csv(f, low_memory=False)
    if "PATNO" in df.columns:
        df["PATNO"] = df["PATNO"].astype(str)
    datasets[name] = df

    inventory_rows.append({
        "dataset": name,
        "file_name": f.name,
        "rows": len(df),
        "columns": df.shape[1],
        "has_PATNO": "PATNO" in df.columns,
        "has_EVENT_ID": "EVENT_ID" in df.columns,
        "unique_PATNO": df["PATNO"].nunique() if "PATNO" in df.columns else np.nan,
        "unique_EVENT_ID": df["EVENT_ID"].nunique() if "EVENT_ID" in df.columns else np.nan,
        "overlap_with_primary_n": len(set(df["PATNO"]).intersection(primary_patnos)) if "PATNO" in df.columns else 0
    })

inventory = pd.DataFrame(inventory_rows).sort_values("dataset")
inventory.to_csv(OUT_DIR / "01_imaging_file_inventory.csv", index=False)
display(inventory)

datscan_names = [n for n in datasets if ("DATSCAN" in n.upper()) or ("DOPAMINE" in n.upper()) or ("SBR" in n.upper())]
print("\nDaTSCAN-related datasets:")
for n in datscan_names:
    print("-", n)


In [ ]:
# ============================================================
# 06. Event overlap for DaTSCAN-related datasets
# ============================================================

event_rows = []

for name in datscan_names:
    df = datasets[name].copy()
    if "PATNO" not in df.columns:
        continue

    df_primary = df[df["PATNO"].isin(primary_patnos)].copy()

    if "EVENT_ID" in df_primary.columns:
        for event_id, g in df_primary.groupby("EVENT_ID", dropna=False):
            event_rows.append({
                "dataset": name,
                "EVENT_ID": event_id,
                "rows": len(g),
                "unique_PATNO": g["PATNO"].nunique(),
                "pct_primary_cohort": round(100 * g["PATNO"].nunique() / len(primary_patnos), 2)
            })
    else:
        event_rows.append({
            "dataset": name,
            "EVENT_ID": "NO_EVENT_ID",
            "rows": len(df_primary),
            "unique_PATNO": df_primary["PATNO"].nunique(),
            "pct_primary_cohort": round(100 * df_primary["PATNO"].nunique() / len(primary_patnos), 2)
        })

event_overlap = pd.DataFrame(event_rows).sort_values(["dataset", "pct_primary_cohort"], ascending=[True, False])
event_overlap.to_csv(OUT_DIR / "02_datscan_event_overlap.csv", index=False)
display(event_overlap.head(50))


In [ ]:
# ============================================================
# 07. Create screening-time Quant SBR feature matrix
# ============================================================

sbr_dataset_name = None
for n in datscan_names:
    if "QUANT_SBR" in n.upper() or "QUANT" in n.upper() and "SBR" in n.upper():
        sbr_dataset_name = n
        break

if sbr_dataset_name is None:
    raise FileNotFoundError("Quant SBR dataset not found.")

sbr = datasets[sbr_dataset_name].copy()
sbr["PATNO"] = sbr["PATNO"].astype(str)

if "EVENT_ID" not in sbr.columns:
    raise ValueError("Quant SBR dataset has no EVENT_ID column.")

# Use SC first because DaTSCAN SBR is mainly captured at screening in PPMI.
sbr_sc = sbr[(sbr["PATNO"].isin(primary_patnos)) & (sbr["EVENT_ID"].astype(str).str.upper() == "SC")].copy()

print("SBR dataset:", sbr_dataset_name)
print("SBR SC rows in primary cohort:", len(sbr_sc))
print("SBR SC unique PATNO:", sbr_sc["PATNO"].nunique())

# Remove exact duplicate rows.
sbr_sc = sbr_sc.drop_duplicates()

# Identify candidate numeric columns.
exclude_cols = {
    "PATNO", "EVENT_ID", "PROTOCOL", "DATSCAN_DATE", "DATSCAN_ANALYZED",
    "DATSCAN_NOT_ANALYZED_REASON", "DATSCAN_OTHER_SPECIFY", "DATSCAN_LIGAND",
    "PREVIOUSLY_ACQUIRED"
}

numeric_cols = []
for c in sbr_sc.columns:
    if c in exclude_cols:
        continue
    converted = pd.to_numeric(sbr_sc[c], errors="coerce")
    if converted.notna().sum() > 0:
        sbr_sc[c] = converted
        numeric_cols.append(c)

# One row per participant; if duplicated, keep the row with the greatest number of non-missing numeric values.
if numeric_cols:
    sbr_sc["_non_missing_numeric"] = sbr_sc[numeric_cols].notna().sum(axis=1)
    sbr_one = (
        sbr_sc.sort_values(["PATNO", "_non_missing_numeric"], ascending=[True, False])
        .drop_duplicates(subset=["PATNO"], keep="first")
        .drop(columns=["_non_missing_numeric"])
    )
else:
    sbr_one = sbr_sc.drop_duplicates(subset=["PATNO"], keep="first")

feature_df = pd.DataFrame({"PATNO": sorted(primary_patnos)})
sbr_features = sbr_one[["PATNO"] + numeric_cols].copy()
sbr_features = sbr_features.rename(columns={c: f"DATSCAN_SBR_SC__{c}" for c in numeric_cols})
feature_df = feature_df.merge(sbr_features, on="PATNO", how="left")

feature_df.to_csv(OUT_DIR / "03_datscan_sbr_screening_feature_matrix.csv", index=False)

print("Feature matrix shape:", feature_df.shape)
display(feature_df.head())


In [ ]:
# ============================================================
# 08. Missingness summary and recommended DaTSCAN features
# ============================================================

feature_cols = [c for c in feature_df.columns if c != "PATNO"]

missingness = []
for c in feature_cols:
    miss_n = feature_df[c].isna().sum()
    missingness.append({
        "feature": c,
        "missing_n": int(miss_n),
        "missing_pct": round(100 * miss_n / len(feature_df), 2),
        "non_missing_n": int(feature_df[c].notna().sum())
    })

missingness_df = pd.DataFrame(missingness).sort_values("missing_pct")
missingness_df.to_csv(OUT_DIR / "04_datscan_sbr_feature_missingness.csv", index=False)

recommended = missingness_df[missingness_df["missing_pct"] <= 30].copy()
recommended.to_csv(OUT_DIR / "05_recommended_datscan_sbr_features_missing_le_30pct.csv", index=False)

print("Total SBR features:", len(feature_cols))
print("Recommended SBR features with missingness <=30%:", len(recommended))
display(missingness_df.head(50))


In [ ]:
# ============================================================
# 09. QC checklist and summary report
# ============================================================

qc_rows = []

qc_rows.append({
    "qc_item": "Imaging ZIP found and extracted",
    "status": "PASS" if len(csv_files) > 0 else "FAIL",
    "details": f"{len(csv_files)} CSV files found."
})

qc_rows.append({
    "qc_item": "Primary analytic cohort loaded",
    "status": "PASS" if len(primary_patnos) == 856 else "CHECK",
    "details": f"{len(primary_patnos)} primary analytic cohort participants."
})

qc_rows.append({
    "qc_item": "Quant SBR dataset found",
    "status": "PASS" if sbr_dataset_name is not None else "FAIL",
    "details": str(sbr_dataset_name)
})

qc_rows.append({
    "qc_item": "SC SBR overlap adequate",
    "status": "PASS" if sbr_sc["PATNO"].nunique() >= 600 else "CHECK",
    "details": f"{sbr_sc['PATNO'].nunique()} participants with SC SBR."
})

qc_rows.append({
    "qc_item": "Recommended DaTSCAN SBR features created",
    "status": "PASS" if len(recommended) > 0 else "FAIL",
    "details": f"{len(recommended)} features with <=30% missingness."
})

qc_rows.append({
    "qc_item": "No modeling performed",
    "status": "PASS",
    "details": "This notebook performs DaTSCAN feature verification only."
})

qc = pd.DataFrame(qc_rows)
qc.to_csv(OUT_DIR / "06_quality_control_checklist.csv", index=False)
display(qc)

summary = f"""Notebook 07b — DaTSCAN Screening-Time Feature Verification
================================================================

Imaging ZIP used:
{IMAGING_ZIP}

Primary analytic cohort participants:
{len(primary_patnos)}

Quant SBR dataset:
{sbr_dataset_name}

SC SBR participants in primary cohort:
{sbr_sc['PATNO'].nunique()}

SBR numeric features:
{len(feature_cols)}

Recommended SBR features with missingness <=30%:
{len(recommended)}

Decision:
DaTSCAN Quant SBR should be reconsidered as a baseline/pre-baseline predictor source using EVENT_ID == 'SC', not only EVENT_ID == 'BL'.
No ML modeling was performed in this notebook.
"""

with open(OUT_DIR / "07_notebook_07b_summary_report.txt", "w") as f:
    f.write(summary)

print(summary)
